# Ансамбль со ссылками (`store_references=True`)

Демонстрирует новый режим `Ensemble`, при котором в пикл-файл ансамбля сохраняются
**строковые ссылки** на группы моделей вместо самих объектов модели.
Загрузка происходит лениво (`resolve_model_reference`) в момент предсказания.

**Сценарий**: данные Титаника разбиваются по классу каюты (`PCLASS`):
- **group_a** — обучен на пассажирах 1-го и 2-го класса (`PCLASS <= 2`)
- **group_b** — обучен на пассажирах 3-го класса (`PCLASS == 3`)

Ансамбль маршрутизирует предсказания по тому же условию.

## 1. Настройка окружения

In [13]:
import os
import sys
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

# Поднимаемся до корня репозитория (папка, где лежит пакет outboxml)
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'outboxml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

# Офлайн MLflow — файловый бэкенд, сервер не нужен
os.environ['MLFLOW_TRACKING_URI'] = 'file:./mlruns_demo'
import mlflow
mlflow.set_tracking_uri('file:./mlruns_demo')

print('Рабочая директория:', REPO_ROOT)

Рабочая директория: C:\Users\dinad\PycharmProjects\outboxml


## 2. Локальный конфиг и рабочие папки

In [14]:
DEMO_PROD = REPO_ROOT / 'demo_ensemble_prod'
DEMO_RESULTS = REPO_ROOT / 'demo_ensemble_results'
DEMO_PROD.mkdir(exist_ok=True)
DEMO_RESULTS.mkdir(exist_ok=True)


class DemoConfig:
    """Локальный конфиг для демо (без MLflow-сервера и PostgreSQL)."""
    work_type_fit = 'CPU'
    work_type_hptune = 'CPU'
    prod_models_path = str(DEMO_PROD)   # os.path.join ожидает str
    results_path = DEMO_RESULTS         # Path, нужен для .mkdir()
    prod_path = REPO_ROOT
    mlflow_tracking_uri = 'file:./mlruns_demo'
    mlflow_experiment = 'EnsembleDemo'
    connection_params = ''
    email_smtp_server = ''
    email_port = ''
    email_sender = ''
    email_login = ''
    email_pass = ''
    email_receivers = []


print('prod_models_path:', DemoConfig.prod_models_path)
print('results_path:    ', DemoConfig.results_path)

prod_models_path: C:\Users\dinad\PycharmProjects\outboxml\demo_ensemble_prod
results_path:     C:\Users\dinad\PycharmProjects\outboxml\demo_ensemble_results


## 3. Загрузка данных и просмотр условия разбивки

In [15]:
DATA_PATH = 'tests/test_data/titanic.csv'
data_all = pd.read_csv(DATA_PATH)

print('Всего строк:', len(data_all))
print()
print('Распределение по PCLASS:')
print(data_all['PCLASS'].value_counts().sort_index().to_frame('count'))
print()
print('Группа A (PCLASS <= 2):', (data_all['PCLASS'] <= 2).sum(), 'строк')
print('Группа B (PCLASS == 3):', (data_all['PCLASS'] == 3).sum(), 'строк')

Всего строк: 891

Распределение по PCLASS:
        count
PCLASS       
1         216
2         184
3         491

Группа A (PCLASS <= 2): 400 строк
Группа B (PCLASS == 3): 491 строк


## 4. Конфигурации для AutoMLManager

In [16]:
MODELS_CFG = {
    'group_name': 'titanic_demo',
    'project': 'titanic',
    'version': '1',
    'data_config': {
        'source': 'csv',
        'local_name_source': DATA_PATH,
        'table_name_source': '',
        'separation': {
            'kind': 'random',
            'random_state': 42,
            'test_train_proportion': 0.2,
            'period_column': ['AGE']
        },
        'extra_columns': ['PASSENGERID', 'SURVIVED', 'PCLASS'],
        'data': {'targetslices': []}
    },
    'models_configs': [
        {
            'name': 'first',
            'column_target': 'SURVIVED',
            'objective': 'poisson',
            'wrapper': 'catboost',
            'relative_features': [],
            'features': [
                {'name': 'SEX', 'encoding': 'to_int', 'default': '0',
                 'replace': {'MALE': '1', 'FEMALE': '2'}},
                {'name': 'AGE',  'default': 0, 'replace': {'_TYPE_': '_NUM_'}},
                {'name': 'SIBSP', 'default': 0, 'replace': {'_TYPE_': '_NUM_'}}
            ]
        },
        {
            'name': 'second',
            'column_target': 'SURVIVED',
            'objective': 'poisson',
            'wrapper': 'catboost',
            'relative_features': [],
            'features': [
                {'name': 'SIBSP', 'default': 0, 'replace': {'_TYPE_': '_NUM_'}},
                {'name': 'PARCH', 'default': 0, 'replace': {'_TYPE_': '_NUM_'}},
                {'name': 'FARE',  'default': 0, 'replace': {'_TYPE_': '_NUM_'}},
                {'name': 'AGE',   'default': 0, 'replace': {'_TYPE_': '_NUM_'}}
            ]
        }
    ]
}

AUTOML_CFG = {
    'group_name': 'titanic_demo',
    'project': 'titanic',
    'version': '1',
    'feature_selection': {
        'top_feautures_to_select': 10,
        'count_category': 100,
        'cutoff_1_category': 0.99,
        'cutoff_nan': 0.7,
        'max_corr_value': 0.8,
        'metric_eval': {'first': 'neg_mean_absolute_error', 'second': 'neg_mean_absolute_error'},
        'cv_diff_value': None,
        'use_temp_data': False,
        'encoding_cat': 'WoE_cat_to_num',
        'encoding_num': 'WoE_num_to_num',
        'default_cat': '_NAN_',
        'default_num': '_MEDIAN_',
        'params': {'iterations': 30},
        'features_to_ignore': ['general']
    },
    'hp_tune': {
        'metric_score': {'first': 'neg_mean_absolute_error', 'second': 'neg_mean_absolute_error'}
    },
    'inference_criteria': {
        'prod_models_folder': 'demo_ensemble_prod',
        'metric_growth_value': {'CompareBusinessMetric': 0.001},
        'threshold': [0.6, 0.8],
        'calculate_threshold': 0,
        'prod_path': None
    },
    'mlflow_experiment': 'EnsembleDemo',
    'grafana_table_name': 'EnsembleDemo',
    'dashboard_name': 'EnsembleDemo',
    'trigger': {'table_name': "public.'TitanicData'", 'field': 'PASSENGERID'}
}

MODELS_CFG_PATH = DEMO_RESULTS / 'models_config.json'
AUTOML_CFG_PATH = DEMO_RESULTS / 'automl_config.json'
MODELS_CFG_PATH.write_text(json.dumps(MODELS_CFG, ensure_ascii=False, indent=2))
AUTOML_CFG_PATH.write_text(json.dumps(AUTOML_CFG, ensure_ascii=False, indent=2))

print('Конфиги записаны в', DEMO_RESULTS)

Конфиги записаны в C:\Users\dinad\PycharmProjects\outboxml\demo_ensemble_results


## 5. Экстрактор с условием отбора данных

In [17]:
from outboxml.automl_manager import AutoMLManager
from outboxml.extractors import Extractor


class FilteredExtractor(Extractor):
    """Экстрактор, фильтрующий данные перед обучением по условию pandas.query."""

    def __init__(self, path: str, query: str):
        self._path = path
        self._query = query
        super().__init__()

    def extract_dataset(self) -> pd.DataFrame:
        df = pd.read_csv(self._path)
        filtered = df.query(self._query).reset_index(drop=True)
        print(f'  Условие «{self._query}»: отобрано {len(filtered)} строк из {len(df)}')
        return filtered

## 6. Обучение группы A — пассажиры 1-го и 2-го класса (`PCLASS <= 2`)

In [18]:
print('=== Группа A: PCLASS <= 2 ===')

automl_a = AutoMLManager(
    auto_ml_config=str(AUTOML_CFG_PATH),
    models_config=str(MODELS_CFG_PATH),
    external_config=DemoConfig,
    extractor=FilteredExtractor(path=DATA_PATH, query='PCLASS <= 2'),
    hp_tune=False,
    retro=False,
)
automl_a.update_models()
results_a = automl_a.get_result()

print('\nОбучены модели (группа A):', list(results_a.keys()))

2026-06-15 17:05:46.234 | DEBUG    | outboxml.datasets_manager:_init_dsmanager:973 - Initializing DSManager


=== Группа A: PCLASS <= 2 ===


2026-06-15 17:05:46.250 | INFO     | outboxml.datasets_manager:__load_all_models_config:859 - All models config from path
2026-06-15 17:05:46.287 | WARNING  | outboxml.datasets_manager:__load_all_models_config:880 - first||File C:\Users\dinad\PycharmProjects\outboxml\demo_ensemble_results\first_v1_subset.pickle already exists. Change version in config file to for new data prepare
2026-06-15 17:05:46.288 | WARNING  | outboxml.datasets_manager:__load_all_models_config:880 - second||File C:\Users\dinad\PycharmProjects\outboxml\demo_ensemble_results\second_v1_subset.pickle already exists. Change version in config file to for new data prepare
2026-06-15 17:05:46.292 | INFO     | outboxml.datasets_manager:__load_all_models_config:887 - Config is loaded
2026-06-15 17:05:46.303 | INFO     | outboxml.datasets_manager:__load_prepare_datasets:918 - Load models prepare datasets
2026-06-15 17:05:46.308 | INFO     | outboxml.datasets_manager:_init_dsmanager:981 - Reading user extractor
2026-06-15 17

  Условие «PCLASS <= 2»: отобрано 400 строк из 891


2026-06-15 17:05:46.844 | INFO     | outboxml.data_subsets:dataset:563 - Reading data from parquet
2026-06-15 17:05:46.855 | DEBUG    | outboxml.data_subsets:_prepare_subset:702 - Model first || Data preparation started
2026-06-15 17:05:46.861 | INFO     | outboxml.core.prepared_datasets:train_test_indexes:284 - Random separation
2026-06-15 17:05:46.941 | INFO     | outboxml.core.prepared_datasets:train_test_indexes:301 - Train: 320, test: 80
2026-06-15 17:05:46.963 | INFO     | outboxml.core.data_prepare:prepare_numerical_feature_series:816 - AGE || Исправлено пропусков: 41
2026-06-15 17:05:46.969 | INFO     | outboxml.core.data_prepare:prepare_dataset:1153 - Find drop values for features
2026-06-15 17:05:46.980 | INFO     | outboxml.core.data_prepare:prepare_dataset:1202 - Feature preparation||Encoding from config to_int
2026-06-15 17:05:46.983 | INFO     | outboxml.core.data_prepare:feature_encoding_series:270 - SEX || Encoding || To int
2026-06-15 17:05:47.003 | DEBUG    | outboxml


Обучены модели (группа A): ['first', 'second']


## 7. Обучение группы B — пассажиры 3-го класса (`PCLASS == 3`)

In [19]:
print('=== Группа B: PCLASS == 3 ===')

automl_b = AutoMLManager(
    auto_ml_config=str(AUTOML_CFG_PATH),
    models_config=str(MODELS_CFG_PATH),
    external_config=DemoConfig,
    extractor=FilteredExtractor(path=DATA_PATH, query='PCLASS == 3'),
    hp_tune=False,
    retro=False,
)
automl_b.update_models()
results_b = automl_b.get_result()

print('\nОбучены модели (группа B):', list(results_b.keys()))

2026-06-15 17:05:50.773 | DEBUG    | outboxml.datasets_manager:_init_dsmanager:973 - Initializing DSManager
2026-06-15 17:05:50.776 | INFO     | outboxml.datasets_manager:__load_all_models_config:859 - All models config from path
2026-06-15 17:05:50.778 | WARNING  | outboxml.datasets_manager:__load_all_models_config:880 - first||File C:\Users\dinad\PycharmProjects\outboxml\demo_ensemble_results\first_v1_subset.pickle already exists. Change version in config file to for new data prepare
2026-06-15 17:05:50.780 | WARNING  | outboxml.datasets_manager:__load_all_models_config:880 - second||File C:\Users\dinad\PycharmProjects\outboxml\demo_ensemble_results\second_v1_subset.pickle already exists. Change version in config file to for new data prepare
2026-06-15 17:05:50.781 | INFO     | outboxml.datasets_manager:__load_all_models_config:887 - Config is loaded
2026-06-15 17:05:50.783 | INFO     | outboxml.datasets_manager:__load_prepare_datasets:918 - Load models prepare datasets
2026-06-15 17

=== Группа B: PCLASS == 3 ===
  Условие «PCLASS == 3»: отобрано 491 строк из 891


2026-06-15 17:05:50.931 | INFO     | outboxml.core.data_prepare:prepare_numerical_feature_series:816 - AGE || Исправлено пропусков: 136
2026-06-15 17:05:50.933 | INFO     | outboxml.core.data_prepare:prepare_dataset:1153 - Find drop values for features
2026-06-15 17:05:50.939 | DEBUG    | outboxml.data_subsets:prepared_subset:1214 - Model second || Data preparation finished
2026-06-15 17:05:50.942 | INFO     | outboxml.core.prepared_datasets:train_test_indexes:284 - Random separation
2026-06-15 17:05:50.946 | INFO     | outboxml.core.prepared_datasets:train_test_indexes:301 - Train: 392, test: 99
2026-06-15 17:05:50.948 | INFO     | outboxml.data_subsets:save_subset_to_pickle:893 - second_v1||Saving subset to pickle
2026-06-15 17:05:50.951 | INFO     | outboxml.data_subsets:save_config_to_pickle:1041 - second_v1_prepare_model_config.pickle||Saving pickle
2026-06-15 17:05:50.954 | INFO     | outboxml.data_subsets:load_subsets_from_pickle:854 - first_v1||Loading subset from pickle
2026-0


Обучены модели (группа B): ['first', 'second']


## 8. Сохранение групп в пикл-файлы

Формат группы: `List[dict]`, где каждый `dict` — результат `DSManagerResult.dict_for_prod_export()`.

Ключи: `model_config`, `model`, `min_max_scaler`, `features_numerical`, `features_categorical`.

In [20]:
def save_group(results: dict, group_name: str, save_dir: Path) -> Path:
    """Сохраняет результаты AutoMLManager как группу моделей (List[dict]) в pickle."""
    group = [r.dict_for_prod_export() for r in results.values()]
    path = save_dir / f'{group_name}.pickle'
    with open(path, 'wb') as f:
        pickle.dump(group, f)
    size_kb = path.stat().st_size / 1024
    print(f'Сохранено: {path.name}  ({len(group)} модели, {size_kb:.1f} КБ)')
    return path


save_group(results_a, 'group_pclass_1_2', DEMO_PROD)
save_group(results_b, 'group_pclass_3',   DEMO_PROD)

print('\nФайлы в prod-папке:')
for p in sorted(DEMO_PROD.iterdir()):
    print(f'  {p.name}')

Сохранено: group_pclass_1_2.pickle  (2 модели, 2007.9 КБ)
Сохранено: group_pclass_3.pickle  (2 модели, 2040.4 КБ)

Файлы в prod-папке:
  group_pclass_1_2.pickle
  group_pclass_3.pickle


## 9. Сборка ансамбля со ссылками (`store_references=True`)

При `store_references=True` третий элемент каждого кортежа в `EnsembleResult.models`
— это **строка** с именем группы, а не объект модели.
Загрузка модели происходит в `resolve_model_reference()` при каждом вызове предсказания.

In [21]:
from outboxml.ensemble import Ensemble

# Условия маршрутизации совпадают с условиями отбора при обучении
ENS_GROUPS = [
    ('PCLASS <= 2', 'group_pclass_1_2'),
    ('PCLASS == 3', 'group_pclass_3'),
]

ens = Ensemble(config=DemoConfig)
ens.make_ensemble(
    ensemble_name='titanic_pclass',
    models_names=['first', 'second'],
    groups=ENS_GROUPS,
    store_references=True,
)

print('Ансамбль создан:', ens._ensemble_name)
print('Модели:', ens._models_names)
print('Флаг is_maked:', ens._is_maked)

2026-06-15 17:05:54.356 | INFO     | outboxml.ensemble:make_ensemble:185 - making ensemble titanic_pclass ...
2026-06-15 17:05:54.378 | INFO     | outboxml.ensemble:make_ensemble:230 - loaded group `group_pclass_1_2`
2026-06-15 17:05:54.387 | INFO     | outboxml.ensemble:make_ensemble:230 - loaded group `group_pclass_3`
2026-06-15 17:05:54.388 | INFO     | outboxml.ensemble:make_ensemble:251 - ensemble titanic_pclass is maked


Ансамбль создан: titanic_pclass
Модели: ['first', 'second']
Флаг is_maked: True


## 10. Проверка структуры — ссылки вместо объектов

In [22]:
print('=== Структура EnsembleResult (store_references=True) ===\n')

for er in ens._result_pickle:
    print(f'Модель: {er.model_name!r}')
    for condition, group_name, ref in er.models:
        print(f'  условие = {condition!r}')
        print(f'  группа  = {group_name!r}')
        print(f'  ref     = {ref!r}  (тип: {type(ref).__name__})')
        print()

print('При store_references=False на месте ref был бы объект CatBoostRegressor.')

=== Структура EnsembleResult (store_references=True) ===

Модель: 'first'
  условие = 'PCLASS <= 2'
  группа  = 'group_pclass_1_2'
  ref     = 'group_pclass_1_2'  (тип: str)

  условие = 'PCLASS == 3'
  группа  = 'group_pclass_3'
  ref     = 'group_pclass_3'  (тип: str)

Модель: 'second'
  условие = 'PCLASS <= 2'
  группа  = 'group_pclass_1_2'
  ref     = 'group_pclass_1_2'  (тип: str)

  условие = 'PCLASS == 3'
  группа  = 'group_pclass_3'
  ref     = 'group_pclass_3'  (тип: str)

При store_references=False на месте ref был бы объект CatBoostRegressor.


## 11. Сохранение ансамбля

In [23]:
ens.save_ensemble(to_mlflow=False)

ensemble_files = sorted(DEMO_RESULTS.glob('titanic_pclass*.pickle'))
print('Сохранённые пикл-файлы ансамбля:')
for p in ensemble_files:
    print(f'  {p.name}  ({p.stat().st_size / 1024:.1f} КБ)')

2026-06-15 17:05:54.492 | INFO     | outboxml.ensemble:save_ensemble:305 - saved ensemble to `titanic_pclass_2026_06_15_17_05_54.pickle`
2026-06-15 17:05:54.494 | INFO     | outboxml.ensemble:save_ensemble:313 - saved ensemble to MLFlow


Сохранённые пикл-файлы ансамбля:
  titanic_pclass_2026_06_15_13_30_52.pickle  (0.2 КБ)
  titanic_pclass_2026_06_15_17_05_54.pickle  (0.2 КБ)


## 12. Верификация: загрузка и проверка пикла

In [24]:
from outboxml.ensemble import EnsembleResult

latest = sorted(DEMO_RESULTS.glob('titanic_pclass*.pickle'))[-1]

with open(latest, 'rb') as f:
    loaded: list = pickle.load(f)

print(f'Загружен: {latest.name}')
print(f'Элементов (по числу model_names): {len(loaded)}')
print()

for item in loaded:
    assert isinstance(item, EnsembleResult), f'Ожидается EnsembleResult, получено {type(item)}'
    print(f'EnsembleResult.model_name = {item.model_name!r}')
    for condition, group_name, ref in item.models:
        assert isinstance(ref, str), f'Ожидается str-ссылка, получено {type(ref).__name__}'
        print(f'  ✓ ref — строка: {ref!r}  →  файл: {DEMO_PROD / (ref + ".pickle")}')
    print()

print('Верификация пройдена: ансамбль хранит строковые ссылки, объекты моделей не загружены.')

Загружен: titanic_pclass_2026_06_15_17_05_54.pickle
Элементов (по числу model_names): 2

EnsembleResult.model_name = 'first'
  ✓ ref — строка: 'group_pclass_1_2'  →  файл: C:\Users\dinad\PycharmProjects\outboxml\demo_ensemble_prod\group_pclass_1_2.pickle
  ✓ ref — строка: 'group_pclass_3'  →  файл: C:\Users\dinad\PycharmProjects\outboxml\demo_ensemble_prod\group_pclass_3.pickle

EnsembleResult.model_name = 'second'
  ✓ ref — строка: 'group_pclass_1_2'  →  файл: C:\Users\dinad\PycharmProjects\outboxml\demo_ensemble_prod\group_pclass_1_2.pickle
  ✓ ref — строка: 'group_pclass_3'  →  файл: C:\Users\dinad\PycharmProjects\outboxml\demo_ensemble_prod\group_pclass_3.pickle

Верификация пройдена: ансамбль хранит строковые ссылки, объекты моделей не загружены.


## Итог

| | `store_references=False` (старый режим) | `store_references=True` (новый режим) |
|---|---|---|
| Что хранится в пикле | Объект модели (CatBoost, XGBoost…) | Строка — имя группы |
| Загрузка при `save_ensemble` | Модели уже в памяти | Модели не загружаются |
| Загрузка при предсказании | Уже загружено | `resolve_model_reference()` читает пикл группы |
| Размер файла ансамбля | Большой (все веса моделей) | Маленький (только строки) |
| Когда использовать | Небольшие модели, нет ограничений памяти | Много групп / большие модели |

При вызове `ensemble_predict()` с ансамблем типа `store_references=True`
функция `resolve_model_reference(ref, model_name, config)` открывает `config.prod_models_path/{ref}.pickle`
и ищет в нём модель с нужным именем.